# Urban Step 3: Train & Fine-Tune Conditioned Trajectory Lane Model

This notebook loads multi-modal JSON trajectory annotations (`(image, route_command, waypoints)`), trains or fine-tunes **ConditionedResNet18Waypoints**, and exports BOTH PyTorch (`urban_conditioned_lane_model.pth`) and ONNX (`urban_conditioned_lane_model.onnx`, Opset 11) for Jetson Nano.

### 1. Setup Environment & Trajectory Dataset Loader

In [ ]:
import os
import sys
import glob
import json
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent.parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

try:
    from jetracer.urban.config import ROUTE_COMMANDS, COMMAND_TO_INDEX, NUM_WAYPOINTS
    from jetracer.urban.lane_model import ConditionedResNet18Waypoints, export_lane_model_to_onnx
    from jetracer.utils import bgr8_to_jpeg
except ImportError:
    from urban.config import ROUTE_COMMANDS, COMMAND_TO_INDEX, NUM_WAYPOINTS
    from urban.lane_model import ConditionedResNet18Waypoints, export_lane_model_to_onnx
    from utils import bgr8_to_jpeg

# Custom PyTorch Dataset for Urban Trajectories
class UrbanTrajectoryDataset(data.Dataset):
    def __init__(self, root_dirs):
        self.samples = []
        for d in root_dirs:
            json_files = glob.glob(os.path.join(d, "*.json"))
            for jf in json_files:
                try:
                    with open(jf, 'r') as f:
                        item = json.load(f)
                    img_p = os.path.join(d, item['image_path'])
                    if os.path.exists(img_p):
                        self.samples.append({
                            'image_path': img_p,
                            'cmd': item['route_command'],
                            'cmd_idx': COMMAND_TO_INDEX.get(item['route_command'], 1),
                            'waypoints': np.array(item['waypoints'], dtype=np.float32) # (5, 2)
                        })
                except Exception:
                    pass
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        cv_img = cv2.imread(s['image_path'])
        if cv_img.shape[0] != 224 or cv_img.shape[1] != 224:
            cv_img = cv2.resize(cv_img, (224, 224))
            
        img_rgb = cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)
        img_norm = (img_rgb.astype(np.float32) / 255.0 - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
        img_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).float()
        
        # Normalize waypoints (0..224) to range [-1, 1]
        wps_norm = (s['waypoints'] / 224.0) * 2.0 - 1.0
        wps_tensor = torch.from_numpy(wps_norm).float()
        
        return img_tensor, s['cmd_idx'], wps_tensor

# Find all urban datasets
dataset_dirs = glob.glob(os.path.join(Path.cwd(), "urban_dataset_*"))
if not dataset_dirs:
    dataset_dirs = glob.glob(os.path.join(parent_dir, "notebooks", "urban", "urban_dataset_*"))

urban_dataset = UrbanTrajectoryDataset(dataset_dirs)
print(f"[*] Loaded Total {len(urban_dataset)} Trajectory Samples across {len(dataset_dirs)} directories.")


### 2. Initialize Model & Check Existing Weights for Fine-Tuning

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pth_save_path  = os.path.join(Path.cwd(), "urban_conditioned_lane_model.pth")
onnx_save_path = os.path.join(Path.cwd(), "urban_conditioned_lane_model.onnx")

model = ConditionedResNet18Waypoints(num_waypoints=NUM_WAYPOINTS, pretrained=True)
model = model.to(device)

is_finetuning = os.path.exists(pth_save_path)
if is_finetuning:
    try:
        model.load_state_dict(torch.load(pth_save_path, map_location=device))
        print(f"[+] FINE-TUNING MODE: Loaded existing weights from '{pth_save_path}'")
    except Exception as e:
        print(f"[!] Could not load weights ({e}). Starting fresh weights.")
        is_finetuning = False
else:
    print(f"[*] NEW MODEL MODE: Initialized with ResNet-18 pretrained weights.")


### 3. Interactive Training UI (`ipywidgets` + Live Sample Display)

In [ ]:
import ipywidgets
from IPython.display import display

epochs_widget     = ipywidgets.IntText(description='epochs', value=10)
batch_size_widget = ipywidgets.IntText(description='batch size', value=8)
loss_widget       = ipywidgets.FloatText(description='loss')
progress_widget   = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')
train_button      = ipywidgets.Button(description='Train & Export ONNX', button_style='warning', icon='play')

sample_preview_widget = ipywidgets.Image(
    format='jpeg', width=224, height=224,
    layout=ipywidgets.Layout(border='2px solid #00ff00', border_radius='4px')
)

def start_training(b):
    if len(urban_dataset) == 0:
        print("[!] Dataset is empty! Annotate samples in urban_02_label_trajectory_dataset.ipynb first.")
        return
        
    epochs = epochs_widget.value
    batch_size = batch_size_widget.value
    train_loader = data.DataLoader(urban_dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    train_button.disabled = True
    model.train()
    start_t = time.time()
    
    print(f"\n[*] Starting Urban Trajectory Training for {epochs} Epochs...")
    for epoch in range(epochs):
        processed = 0
        sum_loss = 0.0
        for images, cmds, waypoints in train_loader:
            images = images.to(device)
            cmds = cmds.to(device)
            waypoints = waypoints.to(device)
            
            optimizer.zero_grad()
            pred_wps = model(images, cmds)
            loss = criterion(pred_wps, waypoints)
            loss.backward()
            optimizer.step()
            
            count = len(cmds)
            processed += count
            sum_loss += float(loss) * count
            
            progress_widget.value = processed / len(urban_dataset)
            loss_widget.value = sum_loss / processed
            
            # Update Live Image Preview
            try:
                img_np = images[0].cpu().numpy().transpose(1, 2, 0)
                img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])) * 255.0
                img_np = np.clip(img_np, 0, 255).astype(np.uint8)
                img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
                
                pts = pred_wps[0].detach().cpu().numpy()
                pts = ((pts + 1.0) / 2.0 * 224.0).astype(int)
                for i in range(len(pts) - 1):
                    cv2.line(img_bgr, tuple(pts[i]), tuple(pts[i+1]), (0, 255, 255), 2)
                for pt in pts:
                    cv2.circle(img_bgr, tuple(pt), 4, (0, 255, 0), -1)
                    
                sample_preview_widget.value = bgr8_to_jpeg(img_bgr)
            except Exception:
                pass
                
        print(f"  Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {loss_widget.value:.4f}")
        
    elapsed = time.time() - start_t
    print(f"[+] Training finished in {elapsed:.1f}s!")
    
    # Save Models
    model.eval()
    torch.save(model.state_dict(), pth_save_path)
    print(f"[+] Saved PyTorch Model -> '{pth_save_path}'")
    export_lane_model_to_onnx(model, onnx_save_path, device=device)
    train_button.disabled = False

train_button.on_click(start_training)

ui_layout = ipywidgets.VBox([
    sample_preview_widget,
    epochs_widget,
    batch_size_widget,
    progress_widget,
    loss_widget,
    train_button
])

display(ui_layout)
